# 03 - Protocolo de evaluación y modelos de referencia

Las funciones están en `src/evaluation.py`. Aquí las validamos con modelos triviales cuyo resultado esperado conocemos, y dejamos una tabla de **referencia mínima** que cualquier modelo real debe superar.

Protocolo: validación cruzada estratificada de **5 folds x 5 repeticiones** sobre train, más el test oficial (5 planetas) con intervalos de confianza por bootstrap estratificado (1.000 remuestras, 95%).

In [1]:
import sys
sys.path.append('..')

import numpy as np
import pandas as pd
from sklearn.base import BaseEstimator, ClassifierMixin
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler, FunctionTransformer
from src.data import load_train_test
from src.preprocessing import preprocess
from src.evaluation import repeated_cv, summarize_cv, report_test

pd.set_option('display.width', 200)
Xtr, ytr, Xte, yte = load_train_test()
Ztr, Zte = preprocess(Xtr), preprocess(Xte)
print('Prevalencia train:', ytr.mean().round(4), '-> un modelo al azar da PR-AUC ~ esta cifra')

Prevalencia train: 0.0073 -> un modelo al azar da PR-AUC ~ esta cifra


## Modelos de referencia

1. **Siempre negativo:** dice "sin planeta" a todo. Muestra por qué accuracy engaña.
2. **Azar:** puntajes aleatorios. Es el piso: PR-AUC ~ prevalencia.
3. **Regresión logística con 3 features simples** (mínimo de la curva, percentil 1, desviación): un modelo muy básico pero que ya usa información real.

In [2]:
class RandomScores(BaseEstimator, ClassifierMixin):
    def fit(self, X, y):
        self.rng_ = np.random.default_rng(0)
        return self
    def predict_proba(self, X):
        p = self.rng_.random(len(X))
        return np.c_[1 - p, p]

def simple_feats(Z):
    return np.c_[Z.min(axis=1), np.percentile(Z, 1, axis=1), Z.std(axis=1)]

modelos = {
    'siempre negativo': lambda: DummyClassifier(strategy='prior'),
    'azar': lambda: RandomScores(),
    'logistica (3 features simples)': lambda: make_pipeline(
        FunctionTransformer(simple_feats), StandardScaler(),
        LogisticRegression(class_weight='balanced', max_iter=1000)),
}

## Validación cruzada (train)

In [3]:
resumen = {}
for nombre, f in modelos.items():
    folds, oof = repeated_cv(f, Ztr, ytr, n_splits=5, n_repeats=5)
    s = summarize_cv(folds)
    resumen[nombre] = {m: f"{s.loc[m, 'mean']:.3f} ± {s.loc[m, 'std']:.3f}" for m in s.index}
tabla_cv = pd.DataFrame(resumen).T
tabla_cv

,recall,precision,f1,pr_auc
siempre negativo,0.000 ± 0.000,0.000 ± 0.000,0.000 ± 0.000,0.007 ± 0.000
azar,0.593 ± 0.027,0.008 ± 0.001,0.016 ± 0.002,0.009 ± 0.001
logistica (3 features simples),0.408 ± 0.185,0.016 ± 0.006,0.032 ± 0.012,0.073 ± 0.068


## Test oficial (5 planetas) con IC 95%

In [4]:
for nombre, f in modelos.items():
    m = f().fit(Ztr, ytr)
    print('==', nombre)
    print(report_test(yte, m.predict_proba(Zte)[:, 1]).round(3), '\n')

== siempre negativo


           valor  ic95_inf  ic95_sup
recall     0.000     0.000     0.000
precision  0.000     0.000     0.000
f1         0.000     0.000     0.000
pr_auc     0.009     0.009     0.009 

== azar


           valor  ic95_inf  ic95_sup
recall     0.400     0.000     0.800
precision  0.006     0.000     0.013
f1         0.013     0.000     0.026
pr_auc     0.008     0.006     0.020 



== logistica (3 features simples)


           valor  ic95_inf  ic95_sup
recall     0.400     0.000     0.800
precision  0.019     0.000     0.042
f1         0.037     0.000     0.081
pr_auc     0.250     0.023     0.807 



## Lectura de los resultados

- **Siempre negativo:** recall 0 y PR-AUC ~0,007 (la prevalencia), aunque su accuracy sería ~99%. Confirma que accuracy no sirve.
- **Azar:** PR-AUC ~0,009, es decir, el piso. Su recall (~0,6) es engañoso: con umbral 0,5 "detecta" más de la mitad de los planetas, pero a costa de marcar la mitad de todas las estrellas (precisión ~0,008).
- **Regresión logística simple:** PR-AUC ~0,07 en CV, unas 10 veces el azar: hay señal incluso con 3 features triviales. Ese es el punto de partida a superar.
- **Incertidumbre del test oficial:** con solo 5 planetas, el IC 95% del PR-AUC de la regresión logística es enorme (aprox. 0,02 a 0,8). Un solo número en el test oficial no permite concluir nada, y por eso el protocolo reporta primero la CV repetida y usa el test oficial solo como confirmación.